In [96]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from langchain.messages import SystemMessage,HumanMessage
from typing import List
class Section(BaseModel):
    name:str=Field(description='Name of this section of the report')
    description:str=Field(description='Brief overview of the main topics and concepts of the section')
class Sections(BaseModel):
    sections:List[Section]=Field(description="Sections of the report")


In [97]:
# building the llm infrastucture
from langchain.chat_models import  init_chat_model
from dotenv import load_dotenv 
import os 
load_dotenv()
llm=init_chat_model("llama-3.1-8b-instant",model_provider="groq",api_key=os.getenv('GROQ'))
planner=llm.with_structured_output(Sections)

In [98]:
# planner.invoke('agentic ai')

In [99]:
from langgraph.types import Send 
import operator
from typing import Annotated
class State(TypedDict):
    topic:str 
    sections:list[Section]
    completed_sections:Annotated[list,operator.add]
    final_report:str
class workerstate(TypedDict):
    section:Section
    

In [100]:
def orchestor(state:State):
    """Orchestor that generates a plan for the report"""
    report_sections=planner.invoke([
        SystemMessage(content="Generate a plan for the report"),
        HumanMessage(content=f"Here is the report topic:{state['topic']}")
    ])
    return{"sections":report_sections.sections}
def llm_call(state:workerstate):
    """worker  writes a section of the report"""
    section=llm.invoke([
SystemMessage(content='write a report section following the provided name and description.'),
HumanMessage(content=f"Here is the section name {state['section'].name} and description:{state['section'].description}")
    ])
    return {"completed_sections":[section.content]}

def assign_worker(state:State):
    """Assign a worker to each section in the plan"""
    return [Send('llm_call',{'section':s}) for s in state['sections']]

def synthesizer(state:State):
    """Synthesize  full report from the sections"""
    completed_section=state['completed_sections']
    completed_report_section="\n\n---\n\n".join(completed_section)
    return {"final_report":completed_report_section}




In [101]:
from langgraph.graph import StateGraph, START, END
graph = StateGraph(State)

# Nodes
graph.add_node("orchestrator", orchestor)
graph.add_node("llm_call", llm_call)
graph.add_node("synthesizer", synthesizer)

# Edges
graph.add_edge(START, "orchestrator")

graph.add_conditional_edges(
    "orchestrator",
    assign_worker
)

graph.add_edge("llm_call", "synthesizer")

graph.add_edge("synthesizer", END)


# ======================
# COMPILE
# ======================

app = graph.compile()

In [102]:
res=app.invoke({"topic":"team iqoo soul bgmi"})

In [103]:
print(res['final_report'])

**Introduction**

The Indian gaming landscape has seen a significant surge in recent years, with several top-tier teams emerging to compete at the highest level. Among these teams, IQOO Soul has carved out a reputation as one of the most formidable forces in the BGMI (Battlegrounds Mobile India) gaming community. This report aims to provide an overview of the team IQOO Soul, its history, and its significance in the BGMI gaming scene.

**Background**

IQOO Soul, previously known as Team Soul, was founded in 2019 by a group of passionate gamers who shared a common goal of competing at the highest level in the Indian gaming scene. The team's early years were marked by a series of modest successes, but it wasn't until 2021 that they began to gain widespread recognition. This was largely due to their impressive performances in various online tournaments, where they consistently demonstrated their skills and teamwork.

**Rise to Prominence**

IQOO Soul's breakthrough moment came in 2022 when

In [106]:
llm.invoke([('user','helo')])

AIMessage(content='Hello. Is there something I can help you with?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 37, 'total_tokens': 49, 'completion_time': 0.14916253, 'completion_tokens_details': None, 'prompt_time': 0.002666527, 'prompt_tokens_details': None, 'queue_time': 0.054866002, 'total_time': 0.151829057}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e6956-18da-7f13-8b82-2dc3d62fcc47-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 12, 'total_tokens': 49})